In [1]:
print("hallow world")

hallow world


In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [3]:
dataset_path = r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx'

In [4]:
new_df = pd.read_excel(r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx')

In [42]:
df = new_df.copy()

In [6]:
consumer_number = df[df['CONSUMER NUMBER'].astype(str).str.match(r'^\d{12}$')]

In [7]:
filter_df = consumer_number[
    ~consumer_number["TWEET/LINK"].astype(str).str.contains("DM", case=False, na=False)
]

In [8]:
filter_df['X_USER_ID'] = filter_df['TWEET/LINK'].str.extract(r'(?:twitter\.com|x\.com)/([^/]+)/')

In [9]:
filter_df['X_USER_ID'] = filter_df['X_USER_ID'].fillna('').apply(lambda x: '@' + x if x and not x.startswith('@') else x)


In [10]:
df['duplicate_case'] = df['REMARKS'].astype(str).str.findall(r'(?i)\b\d{3,5}\b')

In [11]:
df['duplicate_case'] = df['duplicate_case'].astype(str).str.replace(r'[\[\]]', '', regex=True)


In [12]:
df['duplicate_case'] = df['duplicate_case'].astype(str).str.strip("'")

In [13]:
df['duplicate_case'].isnull().value_counts()


duplicate_case
False    32873
Name: count, dtype: int64

In [14]:
df['duplicate_case'].value_counts()


duplicate_case
                         26295
8505                        27
18325', '6920               27
17794', '6287', '2018       24
8724                        20
                         ...  
32862                        1
1468                         1
600                          1
2023', '2024                 1
612                          1
Name: count, Length: 2923, dtype: int64

In [15]:
df['duplicate_case'].count()

np.int64(32873)

In [16]:
# Count how many rows are blank (NaN or empty string)
blank_count = df['duplicate_case'].astype(str).str.strip().replace('', pd.NA).isna().sum()

# Count how many rows are filled (not blank)
filled_count = df['duplicate_case'].astype(str).str.strip().replace('', pd.NA).notna().sum()

# Total rows
total_rows = len(df)

print("Total rows:", total_rows)
print("Filled rows:", filled_count)
print("Blank rows:", blank_count)


Total rows: 32873
Filled rows: 6578
Blank rows: 26295


In [17]:
df['appreciation_tweet'] = df['REMARKS'].astype(str).str.findall(r'(?i)\bappreciation\s*tweet\b')

In [18]:
df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace(r'[\[\]]', ' ', regex=True)


In [19]:
df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace("'", "", regex=False)


In [20]:
df['appreciation_tweet'].astype(str).str.title().value_counts()

appreciation_tweet
                       29429
Appreciation Tweet      3444
Name: count, dtype: int64

In [21]:
df.columns

Index(['SL.NO', 'DATE', 'SHIFT DUTY', 'QUERY/REQUEST/COMPLAINT',
       'COMPLAINT DETAILS', 'COMPLAINT NUMBER', 'SECTION', 'SUB-DIVISION',
       'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'CONSUMER NUMBER',
       'MOBILE NUMB', 'DEPT', 'CLOSED/OPEN', 'REMARKS', 'TWEET/LINK',
       'COMPLAINANT NAME', 'COMPLAINT RECEIVED TIME', 'RESPONSE TIME',
       'SECOND RESPONSE TIME', 'FINAL RESPONSE TIME',
       'FINAL RESPONSE DATE DD/MM/YYYY', 'PSCC/FG/TO', 'ARREARS',
       'REPEAT (Y / N)', 'FORWARDED TO', 'SENTIMENTS', 'NATURE OF TWEET',
       'ACTIONABLE/ NON ACTIONABLE', 'AGEING', 'SLAB', 'SLAB2', 'MINUTE',
       'duplicate_case', 'appreciation_tweet'],
      dtype='object')

In [22]:
remark_df = df[['SL.NO', 'DATE',
                'COMPLAINT DETAILS',
                'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'DEPT', 'CLOSED/OPEN',
                'TWEET/LINK','duplicate_case','appreciation_tweet']]


In [23]:
remark_df.columns

Index(['SL.NO', 'DATE', 'COMPLAINT DETAILS', 'DIVISION', 'CIRCLE',
       'COMPLAINT TYPE', 'DEPT', 'CLOSED/OPEN', 'TWEET/LINK', 'duplicate_case',
       'appreciation_tweet'],
      dtype='object')

In [24]:
remark_df.to_excel('airemk.xlsx')


In [25]:
remark_df.isnull().sum()


SL.NO                    0
DATE                     0
COMPLAINT DETAILS        0
DIVISION              7089
CIRCLE                7139
COMPLAINT TYPE           0
DEPT                     0
CLOSED/OPEN              0
TWEET/LINK               0
duplicate_case           0
appreciation_tweet       0
dtype: int64

In [26]:
selected_month = '2025-8'

In [27]:
def get_month_data(new_df, selected_month):
                """Filter data for selected month"""
                return df[df['DATE'].dt.to_period('M') == selected_month]

In [28]:
month_df = get_month_data(new_df, selected_month)

In [29]:
month_df.shape

(1303, 36)

In [30]:
def get_monthly_remarks_analysis(month_df):
                """Analyze REMARKS column for patterns"""
                temp_df = month_df.copy()
                
                if 'REMARKS' not in temp_df.columns:
                    return {
                        'Appreciation Tweets': 0,
                        'Awaited Consumer': 0,
                        '5-digit Numbers': 0,
                        'Total Remarks': 0
                    }
                
                temp_df['REMARKS'] = temp_df['REMARKS'].fillna('').astype(str)
                
                appreciation_count = temp_df['REMARKS'].str.contains("Appreciation Tweet", case=False, na=False).sum()
                awaited_consumer_count = temp_df['REMARKS'].str.contains("Awaited consumer", case=False, na=False).sum()
                number_count = temp_df['REMARKS'].str.contains(r"\b\d{5}\b", na=False).sum()

                return {
                    'Appreciation Tweets': int(appreciation_count),
                    'Awaited Consumer': int(awaited_consumer_count),
                    '5-digit Numbers': int(number_count),
                    'Total Remarks': len(temp_df)
                }

In [31]:
report = get_monthly_remarks_analysis(month_df)

In [32]:
dfers = pd.DataFrame([report])


In [33]:
dfers

,Appreciation Tweets,Awaited Consumer,5-digit Numbers,Total Remarks
0,154,231,303,1303


In [49]:
import pandas as pd

def get_duplicate_df(df):
    df = df.copy()
    
    # Extract all 3–5 digit numbers from REMARKS
    df['duplicate_case'] = df['REMARKS'].astype(str).str.findall(r'(?i)\b\d{3,5}\b')
    
    # Join list of matches into a single string (comma-separated)
    df['duplicate_case'] = df['duplicate_case'].apply(lambda x: ', '.join(x) if x else '')
    
    # Create a column indicating whether duplicate_case is non-empty
    df['duplicate_case_flag'] = df['duplicate_case'].replace('', pd.NA).notna()
    
    # Extract "appreciation tweet" mentions
    df['appreciation_tweet'] = df['REMARKS'].astype(str).str.findall(r'(?i)\bappreciation\s*tweet\b')
    df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace(r'[\[\]]', ' ', regex=True)
    df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace("'", "", regex=False)
    df['awaited consumer id'] = df['REMARKS'].astype(str).str.contains(r'(?i)\bawaited\s*consumer\b', regex=True)
    
    return df



In [50]:
duplicate_df = get_duplicate_df(df)

In [51]:
duplicate_df.columns

Index(['SL.NO', 'DATE', 'SHIFT DUTY', 'QUERY/REQUEST/COMPLAINT',
       'COMPLAINT DETAILS', 'COMPLAINT NUMBER', 'SECTION', 'SUB-DIVISION',
       'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'CONSUMER NUMBER',
       'MOBILE NUMB', 'DEPT', 'CLOSED/OPEN', 'REMARKS', 'TWEET/LINK',
       'COMPLAINANT NAME', 'COMPLAINT RECEIVED TIME', 'RESPONSE TIME',
       'SECOND RESPONSE TIME', 'FINAL RESPONSE TIME',
       'FINAL RESPONSE DATE DD/MM/YYYY', 'PSCC/FG/TO', 'ARREARS',
       'REPEAT (Y / N)', 'FORWARDED TO', 'SENTIMENTS', 'NATURE OF TWEET',
       'ACTIONABLE/ NON ACTIONABLE', 'AGEING', 'SLAB', 'SLAB2', 'MINUTE',
       'duplicate_case', 'duplicate_case_flag', 'appreciation_tweet',
       'awaited consumer id'],
      dtype='object')

In [52]:
duplicate_df

,SL.NO,DATE,SHIFT DUTY,QUERY/REQUEST/COMPLAINT,COMPLAINT DETAILS,COMPLAINT NUMBER,SECTION,SUB-DIVISION,DIVISION,CIRCLE,...,NATURE OF TWEET,ACTIONABLE/ NON ACTIONABLE,AGEING,SLAB,SLAB2,MINUTE,duplicate_case,duplicate_case_flag,appreciation_tweet,awaited consumer id
0,1,2022-06-10,NaN,NaN,Mr. Mohan Maharana complaint regarding the fun...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1303.0,>180,>365,0.0,,False,,False
1,2,2022-06-10,NaN,NaN,Mr.Bhawanispanda complaint regarding the repla...,4206102206136,SECTION-II,JUNAGARH,KWED,KALAHANDI,...,NaN,NaN,1303.0,>180,>365,0.0,,False,,False
2,3,2022-06-10,NaN,NaN,Mr. Dumb.Orator complaint regarding the unauth...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1303.0,>180,>365,0.0,,False,,False
3,4,2022-06-10,NaN,NaN,"Frequent power interuption At- Sihinapada,Masa...",NaN,NaN,NUAPADA,NUAPADA,KALAHANDI,...,NaN,NaN,1303.0,>180,>365,0.0,,False,,False
4,5,2022-06-10,NaN,NaN,Mr. akshya_patra tweeted that DTR is in unsafe...,4206102207051,SAINTALA,"SDO-2,TITLAGARH",TITLAGARH,BOLANGIR,...,NaN,NaN,1303.0,>180,>365,0.0,,False,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32868,32869,2026-01-02,A,Query,Mr. Abhi28153 tweeted regarding no power suppl...,NaN,UTKELA,KESINGA,KEED,KALAHANDI,...,NaN,NaN,NaN,NaN,NaN,3.0,,False,,False
32869,32870,2026-01-02,A,Query,Mr. Amit Khalkho wrote about name correction r...,NaN,BAMRA,KUCHINDA,JHARSUGUDA,SAMBALPUR,...,NaN,NaN,NaN,NaN,NaN,1.0,32835,True,,False
32870,32871,2026-01-02,B,Complaint,Mr.sanjib kar tweeted regarding no power suppl...,4020126001619,UJALPUR,UJALPUR,SUNDERGARH,ROURKELA,...,NaN,NaN,NaN,NaN,NaN,14.0,,False,,False
32871,32872,2026-01-02,B,Query,Mr. Bikram Keshari Mahanta thanked us.,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,,False,Appreciation Tweet,False
